In [11]:
# pip install torch numpy pandas scikit-learn
from pathlib import Path
from collections import Counter
import random
import re
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

In [2]:
# parameters & get data
DATA_DIR = Path("data")
SEED = 42
MAX_LEN = 512
BATCH_SIZE = 64
EPOCHS = 30
torch.manual_seed(42)

train_data = pd.read_csv(DATA_DIR / "train.csv").sort_values("id")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

In [3]:
# splitting train data set
train, valid = train_test_split(
    train_data,
    test_size=2000,
    random_state=SEED,
    stratify=train_data["label"],
)

print(f"Training: {len(train):,} | Validation: {len(valid):,}")

Training: 8,000 | Validation: 2,000


In [4]:
def tokenize(text):
    text = text.replace("\\n", "\n")
    return re.findall(r"\w+|[^\w\s]", text)[:MAX_LEN]

In [5]:
# build vocab from training set
train_tokens = train["text"].fillna("").map(tokenize)
counts = Counter(token for passage in train_tokens for token in passage)
vocab = {
    word: i
    for i, (word, _) in enumerate(counts.most_common(29_998), start=2)
}

In [6]:
def encode(passages):
    rows = []
    for passage in passages:
        ids = [vocab.get(token, 1) for token in passage] or [1]
        rows.append(ids + [0] * (MAX_LEN - len(ids)))
    return torch.tensor(rows, dtype=torch.long)

In [7]:
train_tokens = train["text"].fillna("").map(tokenize)
valid_tokens = valid["text"].fillna("").map(tokenize)

counts = Counter(token for passage in train_tokens for token in passage)
vocab = {
    token: i
    for i, (token, _) in enumerate(counts.most_common(29_998), start=2)
}

train_inputs = encode(train_tokens)
valid_inputs = encode(valid_tokens)
train_labels = torch.tensor(train["label"].to_numpy(), dtype=torch.float32)

In [8]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, filters):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.conv = nn.Conv1d(
            in_channels=64,
            out_channels=filters,
            kernel_size=3,
            padding=1,
        )
        self.output = nn.Linear(filters, 1)

    def forward(self, ids):
        x = self.embedding(ids).transpose(1, 2)
        x = self.conv(x).relu()
        x = x.masked_fill(ids.unsqueeze(1) == 0, 0)
        x = x.amax(dim=2)
        return self.output(x).squeeze(1)

In [9]:
@torch.no_grad()
def predict(model, inputs):
    model.eval()
    return torch.cat([
        model(batch).sigmoid()
        for batch in inputs.split(BATCH_SIZE)
    ]).numpy()

In [13]:
models = {}
results = []

for filters in [128, 64]:
    torch.manual_seed(SEED)
    model = TextCNN(len(vocab) + 2, filters)

    # same batch
    loader = DataLoader(
        TensorDataset(train_inputs, train_labels),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()

    model.train()
    start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        for ids, labels in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(ids), labels)
            loss.backward()
            optimizer.step()

        if epoch % 5 == 0:
            print(f"{filters} filters model : completed epoch {epoch}/{EPOCHS}")

    training_seconds = time.perf_counter() - start

    results.append({
        "Filters": filters,
        "Train AUROC": roc_auc_score(
            train["label"], predict(model, train_inputs)
        ),
        "Validation AUROC": roc_auc_score(
            valid["label"], predict(model, valid_inputs)
        ),
        "Training seconds": training_seconds,
    })

    models[filters] = model

print("\nResults after 30 epochs")
print(pd.DataFrame(results).round(4).to_string(index=False))

128 filters model : completed epoch 5/30
128 filters model : completed epoch 10/30
128 filters model : completed epoch 15/30
128 filters model : completed epoch 20/30
128 filters model : completed epoch 25/30
128 filters model : completed epoch 30/30
64 filters model : completed epoch 5/30
64 filters model : completed epoch 10/30
64 filters model : completed epoch 15/30
64 filters model : completed epoch 20/30
64 filters model : completed epoch 25/30
64 filters model : completed epoch 30/30

Results after 30 epochs
 Filters  Train AUROC  Validation AUROC  Training seconds
     128          1.0            0.9613          272.1241
      64          1.0            0.9527          162.6318
